# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raju-cse/Flyrank-internship_ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
import duckdb
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
)
""")

print("Connected!")

Connected!


In [14]:
con.sql("""
SHOW TABLES
""").df()

,name


In [15]:
rel = "hf://datasets/FlyRank/internship-warehouse"

tables = [
    "dim_clients",
    "dim_content",
    "fact_content_daily_performance_sample",
    "fact_content_query_90d"
]

for t in tables:
    try:
        print(f"\n===== {t} =====")
        print(con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/{t}/**/*.parquet')").df())
    except Exception as e:
        print(e)


===== dim_clients =====
HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main/dim_clients' (HTTP 404)

===== dim_content =====
HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main/dim_content' (HTTP 404)

===== fact_content_daily_performance_sample =====
HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main/fact_content_daily_performance_sample' (HTTP 404)

===== fact_content_query_90d =====
HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main/fact_content_query_90d' (HTTP 404)


In [16]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT *
FROM glob('{rel}/**/*.parquet')
LIMIT 20
""").df()

,file
0,hf://datasets/FlyRank/internship-warehouse/dim...
1,hf://datasets/FlyRank/internship-warehouse/dim...
2,hf://datasets/FlyRank/internship-warehouse/fac...
3,hf://datasets/FlyRank/internship-warehouse/fac...
4,hf://datasets/FlyRank/internship-warehouse/fac...
5,hf://datasets/FlyRank/internship-warehouse/fac...
6,hf://datasets/FlyRank/internship-warehouse/fac...
7,hf://datasets/FlyRank/internship-warehouse/fac...
8,hf://datasets/FlyRank/internship-warehouse/fac...
9,hf://datasets/FlyRank/internship-warehouse/fac...


In [17]:
files = con.sql("""
SELECT file
FROM glob('hf://datasets/FlyRank/internship-warehouse/**/*.parquet')
""").df()

files

,file
0,hf://datasets/FlyRank/internship-warehouse/dim...
1,hf://datasets/FlyRank/internship-warehouse/dim...
2,hf://datasets/FlyRank/internship-warehouse/fac...
3,hf://datasets/FlyRank/internship-warehouse/fac...
4,hf://datasets/FlyRank/internship-warehouse/fac...
5,hf://datasets/FlyRank/internship-warehouse/fac...
6,hf://datasets/FlyRank/internship-warehouse/fac...
7,hf://datasets/FlyRank/internship-warehouse/fac...
8,hf://datasets/FlyRank/internship-warehouse/fac...
9,hf://datasets/FlyRank/internship-warehouse/fac...


In [18]:
for f in files["file"]:
    print(f)

hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-07/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-08/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_co

In [19]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

print(token[:10] + "...")

hf_iCcbGBB...


In [20]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{token}'
);
""")

print("Secret created")

Secret created


In [21]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

print(token[:12])
print(len(token))

hf_iCcbGBBlP
37


In [26]:
con.sql("""
DESCRIBE
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
LIMIT 5
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


# Section 1: Unit of analysis + time window

# Markdown cell:

One row represents the daily performance record of one content page for one client.

The grain of the dataset is:
(content_hash_id, client_hash_id, report_date)

Observed time window:
The available daily performance data covers January 2025 to June 2026.

This table contains measured search and analytics performance metrics collected over time.

# Section 2: Fields: feature / label / context / excluded

Markdown cell:

Feature:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other
- scroll_events


Label:

- No explicit refresh label exists in the observed dataset.
- A future ML task may define a refresh decision label using historical performance signals.


Context:

- report_date
- client_hash_id
- content_hash_id
- month
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available


Excluded:

- Client identifiers are excluded from modeling because they are identifiers, not performance signals.
- Hash IDs are used only for grouping and joining data.
- Raw private business information is not included.

# Section 3: Verification Queries

In [27]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_contents,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_contents,unique_clients,start_date,end_date
0,78835655,427292,70,2025-01-27,2026-06-30


# 2. Missing values check

In [28]:
con.sql("""
SELECT
    SUM(CASE WHEN content_hash_id IS NULL THEN 1 ELSE 0 END) AS missing_content,
    SUM(CASE WHEN client_hash_id IS NULL THEN 1 ELSE 0 END) AS missing_client,
    SUM(CASE WHEN report_date IS NULL THEN 1 ELSE 0 END) AS missing_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,missing_content,missing_client,missing_date
0,0.0,0.0,0.0


# 3. Grain check

In [29]:
con.sql("""
SELECT
    content_hash_id,
    client_hash_id,
    report_date,
    COUNT(*) AS rows_per_day
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'
)
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,report_date,rows_per_day
0,content_1a0c3a8cc6bdd5bc,client_1a730cb2640a1abf,2026-06-13,2
1,content_8d5a5293688d7d8e,client_1a730cb2640a1abf,2026-06-13,2
2,content_0ffd552a735a66ab,client_1a730cb2640a1abf,2026-06-13,2
3,content_965b9031838c130f,client_1a730cb2640a1abf,2026-06-13,2
4,content_fa351551b6d49870,client_1a730cb2640a1abf,2026-06-13,2
5,content_ff51845ed161bfb4,client_1a730cb2640a1abf,2026-06-13,2
6,content_42239c7e1161beee,client_1a730cb2640a1abf,2026-06-13,2
7,content_0e6cbd7d714dda07,client_1a730cb2640a1abf,2026-06-13,2
8,content_e7c9955db819a140,client_1a730cb2640a1abf,2026-06-13,2
9,content_5ea66e7cd969a440,client_1a730cb2640a1abf,2026-06-13,2


# Section 4: Data limits


Markdown cell:

Data limitations:

- This dataset shows observed performance metrics but cannot explain the reason behind traffic changes.

- Some clients may have incomplete history, creating unbalanced historical observations.

- GSC and GA4 availability depends on whether the client has connected those sources.

- Search performance data alone cannot confirm whether content should actually be refreshed.

- Overlapping time windows can introduce leakage if future information is used during model training.

- Results should be treated as decision-support signals, not guaranteed decisions.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.